# Phase3. Single Factor Testing

## 3.1 Loading Data & Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import re
from pathlib import Path

In [2]:
def load_folder(folder: str | Path, parse_date_index: bool = True, threshold: float = 0.8):
    folder = Path(folder)
    out = {}

    for fp in sorted(folder.glob("*.parquet")):
        df = pd.read_parquet(fp)

        # clean index
        idx_str = df.index.astype(str).str.strip()

        if parse_date_index:
            dt = pd.to_datetime(idx_str, errors="coerce")
            if dt.notna().mean() >= threshold:
                df.index = dt
                df = df.sort_index()
            else:
                df.index = idx_str
        else:
            df.index = idx_str

        out[fp.stem] = df

    return out

In [3]:
PROCESSED_DIR = "/Users/apple/Desktop/PitchBook/Multi-Factor L:S/Processed_Data"

data = load_folder(PROCESSED_DIR)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/1960833517.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(idx_str, errors="coerce")
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/1960833517.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(idx_str, errors="coerce")


In [4]:
industry = pd.read_parquet('/Users/apple/Desktop/PitchBook/Multi-Factor L:S/Processed_Data/industries.parquet')

In [5]:
subindustry = pd.read_parquet('/Users/apple/Desktop/PitchBook/Multi-Factor L:S/Processed_Data/subindustries.parquet')

In [6]:
%run "Factors.ipynb"

Note: you may need to restart the kernel to use updated packages.


## 3.2 Alpha Metrics

In [5]:
# Print Performance Metrics
def performance_summary(pnl: pd.Series, periods_per_year: int = 252):
    pnl = pnl.dropna()
    non_zero = pnl != 0
    if non_zero.any():
        first_idx = pnl[non_zero].index[0]
        pnl = pnl.loc[first_idx:]
    n = len(pnl)
    # cumulative equity (start at 1.0)
    equity = (1 + pnl).cumprod()
    # total return
    total_return = equity.iloc[-1] - 1.0
    # CAGR
    years = n / periods_per_year
    if years > 0:
        cagr = equity.iloc[-1] ** (1 / years) - 1
    else:
        cagr = np.nan
    # annualized volatility
    ann_vol = pnl.std(ddof=1) * np.sqrt(periods_per_year)
    # Sharpe
    sharpe = cagr / ann_vol if pd.notna(ann_vol) and ann_vol != 0 else np.nan
    # drawdowns
    running_max = equity.cummax()
    drawdown = equity / running_max - 1.0
    max_dd = drawdown.min()
    # Calmar
    calmar = cagr / abs(max_dd) if pd.notna(max_dd) and max_dd < 0 else np.nan
    # hit rate
    hit_rate = (pnl > 0).mean()
    print("===== Performance Summary =====")
    print(f"Periods             : {n}")
    print(f"Total Return        : {total_return:8.2%}")
    print(f"CAGR                : {cagr:8.2%}")
    print(f"Ann. Volatility     : {ann_vol:8.2%}")
    print(f"Sharpe Ratio        : {sharpe:8.2f}")
    print(f"Max Drawdown        : {max_dd:8.2%}")
    print(f"Calmar Ratio        : {calmar:8.2f}")
    print(f"Hit Rate (p>0)      : {hit_rate:8.2%}")

In [6]:
# Plot Equity Backtest Curve
def plot_equity_curve(pnl: pd.Series, title: str = "Equity Curve"):
    pnl = pnl.replace([np.inf, -np.inf], np.nan).dropna()
    non_zero = pnl != 0
    if non_zero.any():
        first_idx = pnl[non_zero].index[0]
        pnl = pnl.loc[first_idx:]
        
    equity = (1 + pnl).cumprod()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(equity.index, equity.values)
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Equity (start = 1.0)")
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

## 3.3 Data Pre-Processors

### 3.3.1 Outlier Clipper

In [13]:
def clip_outliers(df, n_std=3.0):
    """
    For each timestamp (row), clip values to mean ± n_std * std.
    """
    out = df.copy().astype(float)
    for time in out.index:
        row = out.loc[time].values.astype(float)
        mask = np.isfinite(row)
        if mask.sum() == 0:
            continue
        vals = row[mask]
        mu = vals.mean()
        sigma = vals.std(ddof=1)
        if sigma == 0 or not np.isfinite(sigma):
            continue
        lo = mu - n_std * sigma
        hi = mu + n_std * sigma
        out.loc[time] = np.clip(row, lo, hi)
    return out

### 3.3.2 Standardization

In [14]:
def standardize(df):
    out = df.copy().astype(float)
    for time in out.index:
        row = out.loc[time].values.astype(float)
        mask = np.isfinite(row)
        if mask.sum() == 0:
            continue
        vals = row[mask]
        mu = vals.mean()
        sigma = vals.std(ddof=1)
        if sigma == 0 or not np.isfinite(sigma):
            out.loc[time, mask] = 0.0
            continue
        z = (row - mu) / sigma
        out.loc[time] = z
    return out


### 3.3.3 Fundamental data processing

## 3.4 Analytics

In [7]:
def _rowwise_corr_nan(X: np.ndarray, Y: np.ndarray, min_n: int = 5) -> np.ndarray:
    """
    Row-wise Pearson corr between X and Y, ignoring NaNs pairwise.
    X, Y: shape (T, N)
    """
    valid = np.isfinite(X) & np.isfinite(Y)
    n = valid.sum(axis=1).astype(float)

    out = np.full(X.shape[0], np.nan, dtype=float)
    ok = n >= max(min_n, 2)
    if not ok.any():
        return out

    # sums
    Xv = np.where(valid, X, 0.0)
    Yv = np.where(valid, Y, 0.0)

    mx = Xv.sum(axis=1) / np.where(n == 0, np.nan, n)
    my = Yv.sum(axis=1) / np.where(n == 0, np.nan, n)

    X0 = np.where(valid, X - mx[:, None], 0.0)
    Y0 = np.where(valid, Y - my[:, None], 0.0)

    # sample covariance / variance
    denom = np.where(n > 1, (n - 1.0), np.nan)
    cov = (X0 * Y0).sum(axis=1) / denom
    vx = (X0 * X0).sum(axis=1) / denom
    vy = (Y0 * Y0).sum(axis=1) / denom

    corr = cov / np.sqrt(vx * vy)
    out[ok] = corr[ok]
    return out


def _winsorize_by_row(df: pd.DataFrame, limits=(0.01, 0.99)) -> pd.DataFrame:
    lo, hi = limits
    q_lo = df.quantile(lo, axis=1)
    q_hi = df.quantile(hi, axis=1)
    return df.clip(lower=q_lo, upper=q_hi, axis=0)


def _zscore_by_row(df: pd.DataFrame) -> pd.DataFrame:
    mu = df.mean(axis=1)
    sd = df.std(axis=1).replace(0.0, np.nan)
    return df.sub(mu, axis=0).div(sd, axis=0)


def _factor_quantiles_by_row(factor: pd.DataFrame, quantiles: int = 5) -> pd.DataFrame:
    """
    Robust quantile binning per date using percentile ranks (handles ties + avoids qcut failures).
    Returns integers 1..q, NaN where factor is NaN.
    """
    q = quantiles
    pct = factor.rank(axis=1, method="first", pct=True)
    bins = np.ceil(pct * q).clip(1, q)
    bins = bins.where(factor.notna())
    return bins.astype("float")  # keep NaN


def compute_alpha_metrics_wide(
    alpha_df: pd.DataFrame,
    close_df: pd.DataFrame,
    periods=(1, 5, 10, 20),
    quantiles: int = 5,
    ic_method: str = "spearman",          # "spearman" (rank IC) or "pearson"
    min_assets: int = 50,
    winsorize_alpha: tuple | None = (0.01, 0.99),
    winsorize_fwdret: tuple | None = None,
    zscore_alpha: bool = False,
    annualization: int = 252,             # for ICIR scaling
    drop_extreme_prices: bool = True,     # treat <=0 as missing
):
    """
    Wide-panel alpha analytics akin to core Alphalens outputs.

    Returns dict with:
      - factor: aligned factor panel (post-processing)
      - prices: aligned close prices
      - fwd_returns: {p: DataFrame}
      - ic: DataFrame (dates x periods)
      - ic_summary: DataFrame (periods x stats)
      - quantiles: DataFrame (dates x assets) with 1..Q
      - qret: {p: DataFrame} (dates x quantiles) mean fwd return per quantile
      - qret_mean: {p: Series} time-average quantile returns
      - spread: {p: Series} (top - bottom) daily
      - spread_summary: DataFrame (periods x stats)
      - coverage: Series (#assets used per date)
    """
    common_dates = alpha_df.index.intersection(close_df.index)
    common_assets = alpha_df.columns.intersection(close_df.columns)
    factor = alpha_df.loc[common_dates, common_assets].astype(float)
    prices = close_df.loc[common_dates, common_assets].astype(float)

    if drop_extreme_prices:
        prices = prices.where(prices > 0)

    # optional preprocessing
    if winsorize_alpha is not None:
        factor = _winsorize_by_row(factor, winsorize_alpha)
    if zscore_alpha:
        factor = _zscore_by_row(factor)

    # coverage diagnostics (per date)
    coverage = factor.notna().sum(axis=1)

    # --- forward returns ---
    fwd_returns = {}
    for p in periods:
        fwd = prices.pct_change(p).shift(-p)
        # mask where factor is missing (like Alphalens factor alignment)
        fwd = fwd.where(factor.notna())
        if winsorize_fwdret is not None:
            fwd = _winsorize_by_row(fwd, winsorize_fwdret)
        fwd_returns[p] = fwd

    # --- IC series ---
    ic = pd.DataFrame(index=factor.index, columns=list(periods), dtype=float)
    X = factor.to_numpy()

    if ic_method.lower() == "spearman":
        X_ic = factor.rank(axis=1).to_numpy()
    elif ic_method.lower() == "pearson":
        X_ic = X
    else:
        raise ValueError("ic_method must be 'spearman' or 'pearson'")

    for p in periods:
        Y = fwd_returns[p].to_numpy()
        if ic_method.lower() == "spearman":
            Y_ic = fwd_returns[p].rank(axis=1).to_numpy()
        else:
            Y_ic = Y

        ic[p] = _rowwise_corr_nan(X_ic, Y_ic, min_n=min_assets)

    # --- IC summary ---
    ic_summary = []
    for p in periods:
        s = ic[p].dropna()
        n = len(s)
        mu = s.mean() if n else np.nan
        sd = s.std(ddof=1) if n > 1 else np.nan
        icir = (mu / sd) * np.sqrt(annualization) if (np.isfinite(mu) and np.isfinite(sd) and sd > 0) else np.nan
        tstat = (mu / (sd / np.sqrt(n))) if (n > 1 and np.isfinite(sd) and sd > 0) else np.nan
        ic_summary.append({
            "period": p,
            "n": n,
            "mean": mu,
            "std": sd,
            "ICIR": icir,
            "tstat": tstat,
            "skew": s.skew() if n > 2 else np.nan,
            "kurt": s.kurt() if n > 3 else np.nan,
            "p05": s.quantile(0.05) if n else np.nan,
            "p50": s.quantile(0.50) if n else np.nan,
            "p95": s.quantile(0.95) if n else np.nan,
        })
    ic_summary = pd.DataFrame(ic_summary).set_index("period")

    # --- factor quantiles + quantile forward returns ---
    q_labels = _factor_quantiles_by_row(factor, quantiles=quantiles)

    qret = {}
    qret_mean = {}
    spread = {}
    spread_summary_rows = []

    q_arr = q_labels.to_numpy()

    for p in periods:
        fwd = fwd_returns[p].to_numpy()
        out = np.full((factor.shape[0], quantiles), np.nan, dtype=float)

        for qi in range(1, quantiles + 1):
            m = (q_arr == qi) & np.isfinite(fwd)
            # nanmean per row
            with np.errstate(invalid="ignore"):
                out[:, qi - 1] = np.where(m, fwd, np.nan).mean(axis=1)

        qret_df = pd.DataFrame(out, index=factor.index, columns=[f"Q{qi}" for qi in range(1, quantiles + 1)])
        qret[p] = qret_df
        qret_mean[p] = qret_df.mean(axis=0, skipna=True)

        spr = qret_df[f"Q{quantiles}"] - qret_df["Q1"]
        spread[p] = spr

        spr_s = spr.dropna()
        n = len(spr_s)
        mu = spr_s.mean() if n else np.nan
        sd = spr_s.std(ddof=1) if n > 1 else np.nan
        sharpe = (mu / sd) * np.sqrt(annualization) if (np.isfinite(mu) and np.isfinite(sd) and sd > 0) else np.nan
        tstat = (mu / (sd / np.sqrt(n))) if (n > 1 and np.isfinite(sd) and sd > 0) else np.nan

        spread_summary_rows.append({
            "period": p,
            "n": n,
            "mean_spread": mu,
            "std_spread": sd,
            "sharpe_spread": sharpe,
            "tstat_spread": tstat,
        })

    spread_summary = pd.DataFrame(spread_summary_rows).set_index("period")

    return {
        "factor": factor,
        "prices": prices,
        "fwd_returns": fwd_returns,
        "ic": ic,
        "ic_summary": ic_summary,
        "quantiles": q_labels,
        "qret": qret,
        "qret_mean": qret_mean,
        "spread": spread,
        "spread_summary": spread_summary,
        "coverage": coverage,
    }

## 3.5 Stock Universe

In [8]:
index_comp = pd.read_csv('/Users/apple/Desktop/PitchBook/Multi-Factor L:S/csi500_comp.csv')
index_weight = pd.read_csv('/Users/apple/Desktop/PitchBook/Multi-Factor L:S/csi500_weights.csv')

In [9]:
def _norm_code(x) -> str:
    """000001.SZ -> 000001 ; 1 -> 000001 ; '000001' -> 000001"""
    s = str(x).strip().upper()
    if "." in s:
        s = s.split(".", 1)[0]
    s = re.sub(r"\D", "", s)
    if len(s) == 0:
        return s
    return s.zfill(6)

def normalize_panel_codes(df: pd.DataFrame, date_col: str | None = None) -> pd.DataFrame:
    """Ensure datetime index (if date_col given or index looks like date) and 6-digit string code columns."""
    out = df.copy()
    if date_col is not None and date_col in out.columns:
        out[date_col] = pd.to_datetime(out[date_col])
        out = out.set_index(date_col)

    # if index is not datetime but looks like date strings, convert
    if not isinstance(out.index, pd.DatetimeIndex):
        try:
            out.index = pd.to_datetime(out.index)
        except Exception:
            pass

    # normalize columns (skip if not code-like)
    new_cols = []
    for c in out.columns:
        # keep non-code columns (rare in your case)
        if isinstance(c, str) and c.lower() in {"date"}:
            new_cols.append(c)
        else:
            new_cols.append(_norm_code(c))
    out.columns = new_cols
    return out

def expand_monthly_to_daily(monthly_df: pd.DataFrame, daily_index: pd.DatetimeIndex) -> pd.DataFrame:
    """Forward-fill monthly values to daily frequency on the daily_index."""
    # IMPORTANT: no backfill (avoids look-ahead before first report)
    return monthly_df.reindex(daily_index, method="ffill")

def align_and_mask_daily_panel(df: pd.DataFrame, daily_index: pd.DatetimeIndex,
                               codes: list[str], comp_daily: pd.DataFrame) -> pd.DataFrame:
    """Reindex to daily_index x codes and mask out non-members -> NaN."""
    out = df.reindex(index=daily_index, columns=codes)
    mask = comp_daily.reindex(index=daily_index, columns=codes).astype("float")
    return out.where(mask == 1)

# ---------- main alignment ----------
def align_data_to_csi500(data: dict,
                         index_comp: pd.DataFrame,
                         index_weight: pd.DataFrame,
                         daily_calendar_source: str = "close",
                         keep_only_common_codes: bool = True,
                         renormalize_daily_weights: bool = True):
    """
    Returns:
      aligned_data: dict of daily panels masked to CSI500 membership
      comp_daily: daily 0/1 membership
      wgt_daily: daily weights aligned (optionally renormalized to sum to 1 over available members)
      codes: final universe code list
    """

    # 1) normalize index panels
    index_comp = normalize_panel_codes(index_comp, date_col="date" if "date" in index_comp.columns else None)
    index_weight = normalize_panel_codes(index_weight, date_col="date" if "date" in index_weight.columns else None)
    index_comp = (index_comp.fillna(0).astype(int))

    # 2) get daily calendar from your price panel
    base = data[daily_calendar_source]
    base = normalize_panel_codes(base)
    daily_index = base.index

    # 3) expand membership/weights monthly -> daily
    comp_daily = expand_monthly_to_daily(index_comp, daily_index).fillna(0).astype(int)
    wgt_daily = expand_monthly_to_daily(index_weight, daily_index)

    # 4) decide universe codes
    index_codes = set(comp_daily.columns)

    if keep_only_common_codes:
        common = index_codes.copy()
        for k, df in data.items():
            if df is None:
                continue
            if isinstance(df, pd.DataFrame):
                df2 = normalize_panel_codes(df) if k not in {"industries", "subindustries"} else df
                common &= set(df2.columns) if isinstance(df2.index, pd.DatetimeIndex) else common
        codes = sorted(common)
    else:
        codes = sorted(index_codes)

    # 5) align + mask each daily panel
    aligned = {}
    for k, df in data.items():
        if df is None:
            continue

        # industries/subindustries are usually code->label tables (not time series)
        if k in {"industries", "subindustries"}:
            # support either: columns=codes (single row) or index=codes
            tmp = df.copy()
            if isinstance(tmp, pd.DataFrame):
                if set(tmp.columns) & set(codes):
                    aligned[k] = tmp.loc[:, [c for c in tmp.columns if c in codes]]
                elif set(tmp.index) & set(codes):
                    aligned[k] = tmp.loc[[c for c in tmp.index if c in codes]]
                else:
                    aligned[k] = tmp
            else:
                aligned[k] = tmp
            continue

        df = normalize_panel_codes(df)
        aligned[k] = align_and_mask_daily_panel(df, daily_index, codes, comp_daily)

    # 6) align weights to the final codes + optionally renormalize
    comp_daily = comp_daily.reindex(index=daily_index, columns=codes).fillna(0).astype(int)
    wgt_daily = wgt_daily.reindex(index=daily_index, columns=codes)

    if renormalize_daily_weights:
        # set non-members to 0 before normalization
        w = wgt_daily.where(comp_daily == 1, 0.0)
        s = w.sum(axis=1).replace(0, np.nan)
        wgt_daily = w.div(s, axis=0)  # sums to 1 across available members
    else:
        wgt_daily = wgt_daily.where(comp_daily == 1)

    return aligned, comp_daily, wgt_daily, codes


# Execute
aligned_data, csi500_comp_daily, csi500_wgt_daily, csi500_codes = align_data_to_csi500(
    data=data,
    index_comp=index_comp,
    index_weight=index_weight,
    daily_calendar_source="close",
    keep_only_common_codes=True,
    renormalize_daily_weights=True,
)

print("Final universe codes:", len(csi500_codes))
print("Daily comp panel:", csi500_comp_daily.shape, "Daily weight panel:", csi500_wgt_daily.shape)
print("Aligned panels:", {k: v.shape for k, v in aligned_data.items() if isinstance(v, pd.DataFrame) and isinstance(v.index, pd.DatetimeIndex)})

Final universe codes: 1771
Daily comp panel: (8592, 1771) Daily weight panel: (8592, 1771)
Aligned panels: {'EBIT': (8592, 1771), 'EBITDA': (8592, 1771), 'cap': (8592, 1771), 'close': (8592, 1771), 'cogs': (8592, 1771), 'debt': (8592, 1771), 'eps': (8592, 1771), 'gross_profit': (8592, 1771), 'high': (8592, 1771), 'low': (8592, 1771), 'net_debt': (8592, 1771), 'net_profit': (8592, 1771), 'open': (8592, 1771), 'operating_income': (8592, 1771), 'pb': (8592, 1771), 'pe': (8592, 1771), 'pe_ttm': (8592, 1771), 'rd_expense': (8592, 1771), 'returns': (8592, 1771), 'revenue': (8592, 1771), 'shares_outstanding': (8592, 1771), 'turnover': (8592, 1771), 'volume': (8592, 1771), 'vwap': (8592, 1771)}


In [10]:
cap_df = aligned_data['cap'] 
close_df = aligned_data["close"] 
open_df = aligned_data['open'] 
high_df = aligned_data['high'] 
low_df = aligned_data['low'] 
volume_df = aligned_data['volume'] 
vwap_df = aligned_data['vwap'] 
returns_df = aligned_data["returns"] 
sharesout_df = aligned_data['shares_outstanding'] 
debt_df = aligned_data['debt'] 
operating_income_df = aligned_data['operating_income']

In [11]:
pe_df = aligned_data['pe'] 
pb_df = aligned_data['pb']

In [12]:
net_profit_df = aligned_data['net_profit']

In [13]:
industry = industry.reindex(close_df.index)
subindustry = subindustry.reindex(close_df.index)

## 3.6 Factor Implementation

In [12]:
res_cap = compute_alpha_metrics_wide(
    alpha_df= cap_df,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_49786/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_49786/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_49786/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [13]:
res_cap['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,4652,-0.015667,0.128805,-1.930896,-8.296185,0.089199,0.495603,-0.221292,-0.018496,0.201366
5,4648,-0.026338,0.140816,-2.969154,-12.751624,0.059210,0.212602,-0.248766,-0.030697,0.209670
10,4643,-0.036177,0.146478,-3.920661,-16.829003,-0.013694,0.094069,-0.274749,-0.040405,0.205585
20,4633,-0.049546,0.150628,-5.221570,-22.388859,-0.066246,-0.018254,-0.295092,-0.054223,0.193252


In [15]:
res_pe = compute_alpha_metrics_wide(
    alpha_df= pe_df,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_49786/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_49786/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_49786/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [16]:
res_pe['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,4652,-0.017920,0.140672,-2.022264,-8.688752,-0.036159,0.583362,-0.258923,-0.015956,0.218024
5,4648,-0.020935,0.146503,-2.268469,-9.742391,-0.183438,0.393092,-0.268637,-0.016164,0.214273
10,4643,-0.024720,0.151099,-2.597055,-11.147572,-0.184106,0.115514,-0.285987,-0.020403,0.214405
20,4633,-0.028133,0.156638,-2.851192,-12.225238,-0.115776,0.073635,-0.297206,-0.025495,0.224332


In [15]:
fwd_ret = close_df.shift(-1) / close_df - 1.0

## 0 Alpha NLTSMOM

In [74]:
alpha0 = alpha_nltsmom_ann(close_df)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/3691097439.py:28: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = close.pct_change()


In [ ]:
res_1 = compute_alpha_metrics_wide(
    alpha_df= alpha1,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

## 1 Alpha 0617a

In [19]:
alpha1 = alpha_0617a(open_df,volume_df)

In [20]:
res_1 = compute_alpha_metrics_wide(
    alpha_df= alpha1,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [22]:
res_1['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,2476,-0.003044,0.062309,-0.775537,-2.430957,0.142314,0.635823,-0.104194,-0.003466,0.101858
5,2472,-0.003239,0.063512,-0.809677,-2.535922,0.066762,0.524996,-0.103082,-0.005201,0.103871
10,2467,-0.005601,0.061215,-1.452403,-4.544347,0.129583,0.581781,-0.102987,-0.007331,0.103688
20,2457,-0.007481,0.062824,-1.890292,-5.902434,0.099898,0.803857,-0.114027,-0.008430,0.099595


## 2 Alpha 0810a

In [23]:
alpha2 = alpha_0810a(returns_df,cap_df,volume_df)

/opt/anaconda3/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [24]:
res_2 = compute_alpha_metrics_wide(
    alpha_df= alpha2,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [25]:
res_2['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,3039,0.004006,0.058877,1.080196,3.751179,0.106936,0.214474,-0.092574,0.002338,0.105654
5,3035,0.002138,0.058325,0.581870,2.019319,0.098825,0.008742,-0.093412,0.001678,0.102375
10,3030,0.001080,0.055932,0.306425,1.062540,0.069063,0.044408,-0.088544,0.001140,0.094125
20,3020,0.000788,0.057904,0.216010,0.747786,0.111022,0.008603,-0.098230,-0.000256,0.100331


## 3 Alpha 0415b

In [26]:
alpha3 = alpha_0415b(close_df,volume_df)

In [27]:
res_3 = compute_alpha_metrics_wide(
    alpha_df= alpha3,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [28]:
res_3['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,2468,-0.006467,0.089823,-1.142961,-3.576875,0.106987,-0.045213,-0.151678,-0.008440,0.143386
5,2464,-0.011999,0.086785,-2.194753,-6.862870,0.154601,0.038026,-0.148327,-0.015587,0.138404
10,2459,-0.015550,0.083480,-2.956892,-9.236649,0.129355,-0.252120,-0.150511,-0.020472,0.126957
20,2449,-0.018705,0.081621,-3.637994,-11.341123,0.251936,-0.351616,-0.143948,-0.025445,0.126145


## 4 Alpha 0616a

In [29]:
alpha4 = alpha_0616a(close_df,open_df,volume_df)

In [30]:
res_4 = compute_alpha_metrics_wide(
    alpha_df= alpha4,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [31]:
res_4['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,2468,-0.007625,0.095096,-1.272770,-3.983111,-0.003410,0.324868,-0.168313,-0.007301,0.145470
5,2464,-0.010988,0.092796,-1.879687,-5.877676,0.063118,0.578564,-0.164749,-0.011978,0.142512
10,2459,-0.017411,0.091329,-3.026304,-9.453478,0.055918,0.261415,-0.167436,-0.018685,0.133030
20,2449,-0.019318,0.093749,-3.271091,-10.197338,0.064634,0.328585,-0.172998,-0.020947,0.141811


## 5 Alpha 0608d

In [17]:
alpha5 = alpha_0608d(close_df,open_df,volume_df,sharesout_df,industry,horro_window=10)

AttributeError: 'DataFrame' object has no attribute 'unique'

In [36]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel7 = alpha7
alpha_panel7 = alpha_panel7.iloc[:-1, :]

# Apply weights and compute pnl
weights7 = alpha_panel7.apply(make_weights, axis=1)
daily_pnl7 = (weights7 * fwd_ret).sum(axis=1)

## 6 Alpha0615a

In [33]:
alpha6 = alpha_0615a(volume_df,sharesout_df,cap_df)

In [34]:
res_6 = compute_alpha_metrics_wide(
    alpha_df= alpha6,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [35]:
res_6['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,3087,-0.003676,0.071123,-0.820498,-2.871744,-0.039093,0.215018,-0.123797,-0.003525,0.115110
5,3083,-0.008033,0.069871,-1.825042,-6.383507,-0.115893,0.186526,-0.127007,-0.005499,0.105040
10,3078,-0.010767,0.066410,-2.573838,-8.995290,0.038589,0.042044,-0.120571,-0.011161,0.101229
20,3068,-0.011756,0.064232,-2.905390,-10.137523,0.078489,0.267492,-0.112438,-0.015504,0.097813


## 7 Alpha0604c

In [36]:
alpha7 = alpha_0604c(close_df,high_df,low_df)

In [37]:
res_7 = compute_alpha_metrics_wide(
    alpha_df= alpha7,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [38]:
res_7['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,4630,0.012002,0.186225,1.023096,4.385374,-0.079973,1.413652,-0.307657,0.011834,0.319330
5,4626,0.021388,0.186544,1.820045,7.798025,0.040715,1.129505,-0.282096,0.016167,0.338849
10,4621,0.024416,0.185321,2.091439,8.955975,0.057673,1.073545,-0.283179,0.022167,0.336821
20,4611,0.036887,0.181748,3.221829,13.781604,0.030693,0.844056,-0.269490,0.032179,0.342309


## 8 Alpha0505

In [39]:
alpha8 = alpha_0505(operating_income_df,vwap_df)

In [40]:
res_8 = compute_alpha_metrics_wide(
    alpha_df= alpha8,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [41]:
res_8['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,2114,0.003493,0.155607,0.356317,1.032020,0.136347,-0.114988,-0.241842,-0.005477,0.257351
5,2110,0.003064,0.156627,0.310559,0.898640,0.139625,-0.335392,-0.240846,-0.005086,0.259804
10,2107,-0.000325,0.153153,-0.033699,-0.097443,0.227147,-0.335127,-0.234655,-0.008856,0.255550
20,2100,-0.007605,0.152134,-0.793558,-2.290805,0.352638,-0.367588,-0.222784,-0.030364,0.258095


## 9 Alpha0413a

In [42]:
alpha9 = alpha_0413a(close_df,high_df,low_df)

In [43]:
res_9 = compute_alpha_metrics_wide(
    alpha_df= alpha9,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [44]:
res_9['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,4631,0.027887,0.132355,3.344796,14.338601,-0.096731,0.561354,-0.190300,0.029404,0.240294
5,4627,0.026462,0.120594,3.483385,14.926257,0.099054,0.906241,-0.162612,0.024267,0.229717
10,4622,0.016029,0.110198,2.309090,9.889071,0.023162,0.794720,-0.158667,0.014818,0.202041
20,4612,0.010657,0.104343,1.621256,6.935792,0.035396,1.145763,-0.159495,0.009050,0.183809


## 10 Alpha0413b

In [45]:
alpha10 = alpha_0413b(operating_income_df,cap_df)

In [46]:
res_10 = compute_alpha_metrics_wide(
    alpha_df= alpha10,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [47]:
res_10['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,4282,0.029286,0.122825,3.785094,15.602697,0.075389,0.555548,-0.173340,0.028431,0.233459
5,4278,0.032205,0.114830,4.452119,18.343697,0.152877,0.701881,-0.151020,0.029515,0.225640
10,4273,0.029569,0.110121,4.262514,17.552216,0.209769,0.657551,-0.146408,0.025419,0.215259
20,4263,0.033299,0.110197,4.796904,19.729606,0.328269,0.721879,-0.139501,0.028091,0.229804


## 11 Alpha0603a

In [48]:
alpha11 = alpha_0603a(volume_df,sharesout_df,cap_df)

In [49]:
res_11 = compute_alpha_metrics_wide(
    alpha_df= alpha11,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [50]:
res_11['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,3093,-0.007453,0.067460,-1.753836,-6.144388,-0.056362,0.184589,-0.116314,-0.006728,0.100657
5,3089,-0.011753,0.065836,-2.833869,-9.921752,-0.140844,0.121396,-0.122449,-0.010694,0.096005
10,3084,-0.014364,0.064568,-3.531418,-12.353957,-0.073277,-0.040860,-0.120544,-0.015206,0.090723
20,3074,-0.015683,0.062686,-3.971438,-13.870735,-0.006882,0.007994,-0.118084,-0.016941,0.085621


## 12 Alpha0416a

In [18]:
alpha12 = alpha_0416a(returns_df,subindustry,volume_df)

AttributeError: 'DataFrame' object has no attribute 'unique'

## 13 Alpha0421b

In [51]:
alpha13 = alpha_0421b(close_df,open_df,volume_df,returns_df)

In [52]:
res_13 = compute_alpha_metrics_wide(
    alpha_df= alpha13,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [53]:
res_13['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,3093,0.004737,0.044368,1.694789,5.937522,0.214346,1.283958,-0.066059,0.003960,0.076907
5,3089,0.003175,0.043360,1.162546,4.070229,-0.040018,0.509440,-0.065685,0.003552,0.074547
10,3084,0.002035,0.043579,0.741147,2.592753,-0.028248,0.611129,-0.068871,0.001049,0.070861
20,3074,0.003172,0.043417,1.159789,4.050706,0.003443,0.510376,-0.067772,0.003154,0.074734


## 14 Alpha0403c

In [54]:
alpha14 = alpha_0403c(volume_df,sharesout_df)

In [55]:
res_14 = compute_alpha_metrics_wide(
    alpha_df= alpha14,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [56]:
res_14['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,3093,-0.007867,0.074242,-1.682106,-5.893089,-0.070636,0.817025,-0.130394,-0.006887,0.111997
5,3089,-0.013971,0.075577,-2.934445,-10.273883,-0.142709,0.334637,-0.138822,-0.012978,0.107010
10,3084,-0.016712,0.076228,-3.480225,-12.174868,-0.107211,0.086756,-0.144965,-0.015560,0.105688
20,3074,-0.017304,0.073462,-3.739205,-13.059632,-0.202804,0.083740,-0.144130,-0.017722,0.099440


## 15 Alpha0618e

In [57]:
alpha15 = alpha_0618e(vwap_df,close_df,volume_df)

In [58]:
res_15 = compute_alpha_metrics_wide(
    alpha_df= alpha15,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [59]:
res_15['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,4652,0.046630,0.136468,5.424198,23.305323,-0.031136,0.930341,-0.179742,0.048197,0.257490
5,4648,0.038283,0.123594,4.917157,21.117710,0.118681,1.169745,-0.166506,0.038382,0.244368
10,4643,0.027960,0.114577,3.873777,16.627757,0.041709,1.376777,-0.157512,0.030316,0.210769
20,4633,0.020905,0.109268,3.037118,13.022445,-0.018742,1.126815,-0.158822,0.022582,0.198467


## 16 Alpha0404b

In [60]:
alpha16 = alpha_0404b(returns_df,operating_income_df,cap_df)

In [61]:
res_16 = compute_alpha_metrics_wide(
    alpha_df= alpha16,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [62]:
res_16['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,3225,0.011780,0.133006,1.405929,5.029536,0.069142,0.939547,-0.207639,0.006439,0.244832
5,3221,0.015992,0.131068,1.936885,6.924667,0.028073,0.650880,-0.198300,0.014308,0.236368
10,3216,0.014521,0.131955,1.746901,6.240597,0.152717,0.994181,-0.197753,0.011465,0.236170
20,3206,0.012553,0.133055,1.497697,5.342019,0.285840,0.832849,-0.196387,0.007077,0.241351


## 17 Alpha0412

In [63]:
alpha17 = alpha_0412(operating_income_df,vwap_df,volume_df,returns_df)

In [64]:
res_17 = compute_alpha_metrics_wide(
    alpha_df= alpha17,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [66]:
res_17['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,1999,0.016995,0.111271,2.424529,6.828630,0.109425,0.279790,-0.163429,0.014803,0.195585
5,1995,0.022552,0.108835,3.289414,9.255284,0.018257,-0.048866,-0.156991,0.023746,0.210444
10,1990,0.022716,0.108205,3.332676,9.365249,-0.056993,-0.053089,-0.163158,0.024688,0.196176
20,1980,0.021462,0.105115,3.241184,9.085232,-0.064714,0.137320,-0.156620,0.023743,0.190018


## 18 Alpha0418b

In [67]:
alpha18 = alpha_0418b(operating_income_df,vwap_df)

In [68]:
res_18 = compute_alpha_metrics_wide(
    alpha_df= alpha18,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_22638/4221118392.py:109: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

In [70]:
res_18['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,3093,0.019928,0.109998,2.875971,10.075678,0.154081,1.114778,-0.160987,0.018132,0.204700
5,3089,0.024216,0.106249,3.618003,12.667112,0.193283,0.646006,-0.142516,0.021040,0.204184
10,3084,0.025331,0.105666,3.805607,13.313151,0.304588,0.772691,-0.139622,0.019737,0.206236
20,3074,0.027652,0.106106,4.137093,14.449306,0.360736,0.751784,-0.135833,0.021380,0.214558


## 19 Industry PE Profit Gate